# Connecting ferry components for Norway

This notebook is the Norway version of `connect.ipynb`. It connects ferry geometries to the compact driving road graph, generates synthetic ferry edges, and writes Norway-specific Pandana node and edge tables.

Expected local inputs under `./data/`:

- `no_nodes_and_edges_driving.pkl`: tuple `(no_nodes, no_edges)` from the compact Norway driving road-network build.
- `ferries_no.pkl`: Norway ferry geometries with an `id` column and, where available, `name`, `from`, and `to` columns.

The notebook uses `EPSG:25833` as the projected CRS for nearest-road and ferry-length calculations. This is a practical national CRS for mainland Norway; if the analysis includes Svalbard or far offshore routes, confirm the projection before interpreting metric distances.


In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────────
import json
import pickle
from pathlib import Path
from time import perf_counter as pc

# ─── Scientific Computing ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Geospatial Processing ──────────────────────────────────────────────────────
import geopandas as gpd
import pyproj
from scipy.spatial import cKDTree
from shapely.geometry import LineString, MultiLineString, Point

# ─── Mapping and Visualization ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import folium

# ─── Custom Modules ────────────────────────────────────────────────────────────
import geo_util as gu

In [ ]:
# global definitions
country_name = 'Norway'
country_prefix = 'no'
projected_crs = 'EPSG:25833'
_data_path = Path('./data/')


In [ ]:
required_inputs = [
    _data_path / 'no_nodes_and_edges_driving.pkl',
    _data_path / 'ferries_no.pkl',
]
missing_inputs = [path for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        'Missing Norway connect inputs: ' + ', '.join(str(path) for path in missing_inputs)
    )


In [ ]:
nodes_and_edges_path = _data_path.joinpath('no_nodes_and_edges_driving').with_suffix('.pkl')

with open(nodes_and_edges_path, 'rb') as f:
    (no_nodes, no_edges) = pickle.load(f)

## Drivable-road component audit

The Norway drivable road graph is much larger than the Netherlands example. The connected-component count and map are therefore computed with a sparse graph workflow outside the interactive notebook, then loaded here as reproducible artefacts. This avoids constructing a NetworkX graph with more than eighteen million nodes in the notebook kernel.


In [ ]:
# Norway is too large for the notebook-local NetworkX component helper.
# Use the precomputed sparse-graph audit produced from no_nodes_and_edges_driving.pkl.
component_summary_path = _data_path / 'no_drivable_components_summary.json'
component_figure_path = Path(r'C:\local\temp\HandsOnBook\generated-figures\Norway\norway_drivable_components.png')

with open(component_summary_path, 'r', encoding='utf-8') as f:
    component_summary = json.load(f)

print(f"Drivable connected components: {component_summary['connected_components']:,}")
print(f"Largest component nodes: {component_summary['largest_component_nodes']:,}")
print(f"Largest component share: {component_summary['largest_component_share']:.2%}")


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(component_figure_path)))


In [ ]:
ferries_path = _data_path.joinpath('ferries_no.pkl')

with open(ferries_path, 'rb') as f:
    no_ferries = pickle.load(f)

In [ ]:
no_ferries_projected = no_ferries.to_crs(projected_crs)
no_nodes_projected = no_nodes.to_crs(projected_crs)

In [ ]:
no_node_kdtree, no_node_coords, no_node_ids = gu.buildRoadKDTree(no_nodes_projected)

In [ ]:
no_ferry_coordinates, no_ferry_ids = gu.extractFerryCoordinatesAndIds(no_ferries_projected)

In [ ]:
no_connectedFerries = gu.connectFerryToRoad(
    ferryCoords=no_ferry_coordinates,
    ferryIds=no_ferry_ids,
    roadCoords=no_node_coords,
    roadNodeIds=no_node_ids,
    roadTree=no_node_kdtree,
    maxDistance=5000.0,
    targetCRS=no_ferries.crs,
    sourceCRS=no_ferries_projected.crs
)

In [ ]:
no_connectedFerries.ferryId.value_counts()

In [ ]:
no_ferries_projected[
    no_ferries_projected.id == no_connectedFerries.ferryId.value_counts().idxmin()
].explore(
    tooltip=['id', 'name'],
    style_kwds={'color': 'red', 'weight': 8}
)

In [ ]:
m = no_ferries.explore(
    color='blue',
    tooltip=['name', 'from', 'to'],
    style_kwds={'weight': 2}
)
no_connectedFerries[no_connectedFerries['distance'] <=0.5].explore( 
    m=m,
    color='red',
    tooltip=['ferryPoint', 'roadPoint', 'distance'],
    style_kwds={'weight': 8}
)
m

In [ ]:
coord_to_ferry_ids = gu.groupFerryIdsByCoordinate(no_ferry_coordinates, no_ferry_ids)

# Extract the lengths of lists
list_lengths = [len(ferry_ids) for ferry_ids in coord_to_ferry_ids.values()]

# Plotting the histogram
plt.hist(list_lengths, bins=range(1, max(list_lengths) + 2), edgecolor='black')
plt.xlabel('Number of Ferry IDs per Coordinate')
plt.ylabel('Frequency')
plt.title('Histogram of Ferry IDs List Lengths')
plt.show()

In [ ]:
# Identify the maximum length
max_length = max(list_lengths)

# Find the coordinate(s) with the longest list of ferry IDs
coords_with_max_length = [coord for coord, ids in coord_to_ferry_ids.items() if len(ids) == max_length]

# Extract and print the ferry IDs associated with the longest list
longest_ferry_id_lists = [coord_to_ferry_ids[coord] for coord in coords_with_max_length]

In [ ]:
max_length

In [ ]:
no_ferries[no_ferries.id.isin(longest_ferry_id_lists[0])]

In [ ]:
# Transformer from the Norway projected CRS to EPSG:4326
to_wgs84 = pyproj.Transformer.from_crs(projected_crs, 'EPSG:4326', always_xy=True).transform

m = no_ferries[no_ferries.id.isin(longest_ferry_id_lists[0])].explore(
    tooltip=['id', 'name'],
    style_kwds={'weight': 2},
    color='blue',
    legend=False
)
for coord in coords_with_max_length:
    # Transform from (x, y) in meters to (lon, lat) in degrees
    lon, lat = to_wgs84(*coord)

    print(f"Coordinate: ({lat},{lon}), Ferry IDs: {coord_to_ferry_ids[coord]}")
    folium.Marker(
        location=(lat, lon),
        popup=f"Ferry IDs: {', '.join(map(str, coord_to_ferry_ids[coord]))}",
        icon=folium.Icon(color='red')
    ).add_to(m)
m

In [ ]:
ferry_edges = gu.generateZeroDistanceEdges(no_connectedFerries)
ferry_edges['distance_m'] = ferry_edges['ferryId'].map( no_ferries_projected.set_index('id').geometry.length.to_dict() )
no_ferries[no_ferries.id == ferry_edges.loc[ferry_edges.distance_m.idxmax()].ferryId].explore()

In [ ]:
nodes_df = no_nodes[['id', 'geometry']].copy().rename(columns={'id': 'osm_id'})

# Create full list of node ids used
used_node_ids = pd.Index(pd.concat([no_edges['u'], no_edges['v'], ferry_edges['fromNodeId'], ferry_edges['toNodeId']]).unique())

missing_node_ids = set(used_node_ids) - set(nodes_df['osm_id'])
assert not missing_node_ids, f'Missing node geometries for {len(missing_node_ids)} used edge endpoints'

# Mapping
id_map = {nid: i for i, nid in enumerate(used_node_ids)}

nodes_df = nodes_df[nodes_df['osm_id'].isin(id_map)].copy()
nodes_df['id'] = nodes_df['osm_id'].map(id_map)

nodes_df = nodes_df.set_index('id',drop=False)

In [ ]:
nodes_df.shape, no_nodes.shape

In [ ]:
land_edges = no_edges[['u', 'v', 'length']].copy()
land_edges['from'] = land_edges['u'].map(id_map)
land_edges['to'] = land_edges['v'].map(id_map)
land_edges['weight'] = land_edges['length']

ferry_edges = ferry_edges[['fromNodeId', 'toNodeId', 'distance_m']].copy()
ferry_edges['from'] = ferry_edges['fromNodeId'].map(id_map)
ferry_edges['to'] = ferry_edges['toNodeId'].map(id_map)
ferry_edges['weight'] = ferry_edges['distance_m']

In [ ]:
edges_df = pd.concat(
    [   land_edges[['from', 'to', 'weight']], 
        ferry_edges[['from', 'to', 'weight']]
    ]
).rename(columns={'from': 'u', 'to': 'v', 'weight': 'length'})

In [ ]:
assert set(nodes_df.id) == ( set(edges_df.u) | set(edges_df.v) )

In [ ]:
nodes_df = gu.assignComponentsToNodes(nodes_df, edges_df)

In [ ]:
nodes_df['country'] = country_name
edges_df['country'] = country_name

nodes_df.to_parquet(_data_path.joinpath('no_pandana_nodes.parquet'), index=False)
edges_df.to_parquet(_data_path.joinpath('no_pandana_edges.parquet'), index=False)

In [ ]:
gu.showNodesColoredPerComponentWithBasemap(nodes_df, width=800, height=600, file_name='after_connecting_norway_components_with_ferries.png')